# Retention Copilot: run the agent and the evaluation on Kaggle

Settings (right panel): Accelerator = *GPU T4 x2*, *Internet = On*. Run the cells ONE AT A TIME (never 'Run All').

1-2 install and start the model. 3 gets the code. 4 smoke test. 5-6 DEVELOPMENT runs on the dev pool. 7-9 the FINAL evaluation on the held-out pool (prompts are frozen). 10 package results.

In [ ]:
GITHUB_USER = "Randeep-Sidhu"
MODEL = "granite4:micro"
BIG = "granite4:small-h"

# Install Ollama (Internet must be On)
!apt-get install -y -q zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the server in the background and download the small model
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
!ollama pull {MODEL}

In [ ]:
# Get the code (works whether or not the repo has a nested folder), install extras, run the offline tests
import os, pathlib
%cd /kaggle/working
!rm -rf retention-copilot
!git clone https://github.com/{GITHUB_USER}/Retention-copilot.git retention-copilot
root = next(p.parent.parent for p in pathlib.Path("retention-copilot").rglob("src/agent.py"))
os.chdir(root)
print("project root:", root)
!pip install -q langgraph langchain-ollama fastembed pytest
!python -m pytest -q 2>&1 | tail -5

In [ ]:
# Smoke test (development customers)
!python -m src.agent --customer auto --n 3 --config rag+verify --no-persist --model {MODEL}

In [ ]:
# DEVELOPMENT run: small model, 16 dev customers
!python -m src.evaluate --pool dev --n 16 --model {MODEL} --configs none,rag,rag+verify --fresh
!sed -n '/## Results/,/## Grounding/p' reports/dev_eval_granite4-micro.md

In [ ]:
# DEVELOPMENT run: larger model (~19 GB download; only the last line of the download is shown)
!ollama pull {BIG} 2>&1 | tail -1
!python -m src.evaluate --pool dev --n 16 --model {BIG} --configs none,rag,rag+verify
!sed -n '/## Results/,/## Grounding/p' reports/dev_eval_granite4-small-h.md

In [ ]:
# FINAL evaluation 1 of 2: IBM Granite 4 small (32B MoE) on the held-out pool. ~45-60 min. Resumable: re-run if the session drops.
!ollama pull {BIG} 2>&1 | tail -1
!python -m src.evaluate --pool final --n 48 --model {BIG} --configs none,full,rag,rag+verify

In [ ]:
# FINAL evaluation 2 of 2: Granite 4 micro (3B) on the same customers. ~20-25 min.
!ollama pull {MODEL} 2>&1 | tail -1
!python -m src.evaluate --pool final --n 48 --model {MODEL} --configs none,full,rag,rag+verify

In [ ]:
# Side-by-side comparison of both models
!python -m src.compare_models

In [ ]:
# Package the results, then download results.zip from the Output panel (/kaggle/working)
import zipfile, pathlib
root = pathlib.Path.cwd()
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [*root.glob("db/*_runs.db"), *root.glob("reports/**/*")]:
        if p.is_file():
            z.write(p, p.relative_to(root))
print("results.zip written")